In [ ]:
import cv2
import numpy as np
import tensorflow as tf

MODEL_PATH = "../models/face_mask_detection_model.keras"
INVERT_FOR_GRAY = True
ROI_REL = (0.35, 0.15, 0.30, 0.60)  # (x, y, w, h) relatif
IMG_SIZE = (128, 128)
labels = ["mask_weared_incorrect", "with_mask", "without_mask"]

def preprocess(image, H, W, Convert):
    if Convert:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (5, 5), 0)
        _, bw = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
        bw = cv2.resize(bw, (W, H))
       
        bw = cv2.cvtColor(bw, cv2.COLOR_GRAY2BGR)
        return bw
    else:
        img = cv2.resize(image, (W, H))
        return img


In [2]:
def main():
    # Charger le modèle avec gestion d'erreur
    try:
        model = tf.keras.models.load_model(MODEL_PATH)
        print("Modèle chargé avec succès")
    except Exception as e:
        print(f"Erreur lors du chargement du modèle: {e}")
        return

    cap = cv2.VideoCapture(0)  # ouvrir webcam
    
    if not cap.isOpened():
        print("Erreur: impossible d'ouvrir la webcam")
        return

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Erreur: impossible de lire le frame")
            break

        Hf, Wf = frame.shape[:2]
        x, y, w, h = ROI_REL
        x, y, w, h = int(x*Wf), int(y*Hf), int(w*Wf), int(h*Hf)
        roi = frame[y:y+h, x:x+w]

        # Vérifier que ROI n'est pas vide
        if roi.size == 0:
            print("ROI vide, continuez...")
            continue

        # Prétraitement
        processed = preprocess(roi, IMG_SIZE[0], IMG_SIZE[1], INVERT_FOR_GRAY)
        processed = processed.astype("float32") / 255.0
        processed = np.expand_dims(processed, axis=0)

        try:
            # Prediction avec gestion d'erreur
            preds = model.predict(processed, verbose=0)
            cls = np.argmax(preds)
            confidence = np.max(preds)
            label = labels[cls]
            
            # Afficher la confiance
            display_text = f"{label} ({confidence:.2f})"
            
        except Exception as e:
            print(f"Erreur lors de la prédiction: {e}")
            display_text = "Erreur prédiction"

        # Affichage avec couleur selon la prédiction
        color = (0, 255, 0) if cls == 1 else (0, 0, 255)  # Vert pour masque, rouge sinon
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, display_text, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        cv2.imshow("Live Detection", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):  # presser q pour quitter
            break

    cap.release()
    cv2.destroyAllWindows()

In [3]:
if __name__ == "__main__":
    main()

Modèle chargé avec succès
